# Notebook 3: Extracción y Preprocesamiento - Guías CANMAT (Trastornos del Estado de Ánimo)

Este notebook procesa las guías clínicas de la **CANMAT** (*Canadian Network for Mood and Anxiety Treatments* / ISBD) para **Depresión Mayor** y **Trastorno Bipolar**.

### Objetivos:
1. Extraer las recomendaciones clínicas estructuradas por jerarquía terapéutica (1ª línea, 2ª línea, 3ª línea, Potenciación).
2. Capturar los **Niveles de Evidencia científica** (Level 1, 2, 3) asignados por los consensos internacionales.
3. Modelar las reglas de prescripción y evitar tratamientos contraindicados (ej. evitar antidepresivos en monoterapia en depresión bipolar).
4. Generar la base de conocimiento clínico que alimentará el motor de recomendaciones y el RAG en Streamlit.


In [ ]:
import os
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f'Entorno configurado correctamente. Raíz del proyecto: {project_root}')


## 1. Corpus de Recomendaciones CANMAT

Estructuramos las tablas de evidencia de las guías CANMAT (consenso de adultos para depresión y bipolaridad).


In [ ]:
DATOS_CANMAT_RAW = [
    # Trastorno Depresivo Mayor (TDM) - Monoterapia Inicial
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Citalopram", "atc": "N06AB04", "nivel_evidencia": "Level 1", "comentarios": "ISRS de alta tolerabilidad; vigilar intervalo QTc en dosis altas (>40 mg)."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Escitalopram", "atc": "N06AB10", "nivel_evidencia": "Level 1", "comentarios": "ISRS con menor perfil de interacciones citocromo P450."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Sertralina", "atc": "N06AB06", "nivel_evidencia": "Level 1", "comentarios": "ISRS seguro en pacientes con cardiopatía isquémica."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Venlafaxina", "atc": "N06AX16", "nivel_evidencia": "Level 1", "comentarios": "IRSN; monitorizar presión arterial sistólica en dosis elevadas."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Duloxetina", "atc": "N06AX21", "nivel_evidencia": "Level 1", "comentarios": "IRSN con beneficio adicional en dolor neuropático crónico."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Bupropión", "atc": "N06AX12", "nivel_evidencia": "Level 1", "comentarios": "IRND sin disfunción sexual asociada; contraindicado en epilepsia y bulimia."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Vortioxetina", "atc": "N06AX26", "nivel_evidencia": "Level 1", "comentarios": "Modulador multimodal con beneficio en disfunción cognitiva."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Segunda Línea", "estrategia": "Monoterapia", "farmaco": "Amitriptilina", "atc": "N06AA09", "nivel_evidencia": "Level 1", "comentarios": "Tricíclico; alta eficacia pero menor tolerabilidad por efectos anticolinérgicos y cardiotoxicidad."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Aguda", "linea": "Segunda Línea", "estrategia": "Monoterapia", "farmaco": "Quetiapina", "atc": "N05AH04", "nivel_evidencia": "Level 2", "comentarios": "Antipsicótico atípico con propiedades antidepresivas aprobadas."},
    
    # TDM - Estrategias de Potenciación (Add-on)
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Resistencia / Potenciación", "linea": "Primera Línea", "estrategia": "Potenciación (Add-on)", "farmaco": "Aripiprazol", "atc": "N05AX12", "nivel_evidencia": "Level 1", "comentarios": "Añadido al antidepresivo; dosis bajas (2-5 mg)."},
    {"patologia": "Trastorno Depresivo Mayor (TDM)", "fase": "Resistencia / Potenciación", "linea": "Primera Línea", "estrategia": "Potenciación (Add-on)", "farmaco": "Litio (Carbonato)", "atc": "N05AN01", "nivel_evidencia": "Level 1", "comentarios": "Estrategia clásica con reducción probada de riesgo suicida."},
    
    # Trastorno Bipolar - Manía Aguda
    {"patologia": "Trastorno Bipolar", "fase": "Manía Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Litio (Carbonato)", "atc": "N05AN01", "nivel_evidencia": "Level 1", "comentarios": "Piedra angular en manía euforica clásica; control de litemia."},
    {"patologia": "Trastorno Bipolar", "fase": "Manía Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Divalproato de Sodio", "atc": "N03AG01", "nivel_evidencia": "Level 1", "comentarios": "Eficaz en episodios mixtos y cicladores rápidos. Teratogénico."},
    {"patologia": "Trastorno Bipolar", "fase": "Manía Aguda", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Quetiapina", "atc": "N05AH04", "nivel_evidencia": "Level 1", "comentarios": "Monoterapia atípica de rápida titulación."},
    {"patologia": "Trastorno Bipolar", "fase": "Manía Aguda", "linea": "Primera Línea", "estrategia": "Combinación", "farmaco": "Litio + Antipsicótico Atípico", "atc": "N/D", "nivel_evidencia": "Level 1", "comentarios": "Mayor tasa de respuesta en manía severa que monoterapia."},
    
    # Trastorno Bipolar - Depresión Bipolar
    {"patologia": "Trastorno Bipolar", "fase": "Depresión Bipolar", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Quetiapina", "atc": "N05AH04", "nivel_evidencia": "Level 1", "comentarios": "Dosis 300 mg/día; robusta evidencia en depresión bipolar I y II."},
    {"patologia": "Trastorno Bipolar", "fase": "Depresión Bipolar", "linea": "Primera Línea", "estrategia": "Monoterapia", "farmaco": "Lurasidona", "atc": "N05AE05", "nivel_evidencia": "Level 1", "comentarios": "Bajo impacto metabólico; administrar con alimentos."},
    {"patologia": "Trastorno Bipolar", "fase": "Depresión Bipolar", "linea": "Primera Línea", "estrategia": "Combinación", "farmaco": "Litio + Lamotrigina", "atc": "N/D", "nivel_evidencia": "Level 2", "comentarios": "Sinérgica para prevención y tratamiento de fases depresivas."},
    {"patologia": "Trastorno Bipolar", "fase": "Depresión Bipolar", "linea": "No Recomendado / Contraindicado", "estrategia": "Monoterapia", "farmaco": "Antidepresivos ISRS en Monoterapia", "atc": "N06AB", "nivel_evidencia": "Level 1 (Daño)", "comentarios": "ALERTA: Alto riesgo de viraje maníaco y desestabilización a ciclos rápidos."}
]

df_canmat = pd.DataFrame(DATOS_CANMAT_RAW)
print(f"Total de directrices CANMAT ingresadas: {len(df_canmat)}")
df_canmat.head(6)


## 2. Limpieza, Validación y Normalización de Categorías

Estandarizamos los textos y validamos las clasificaciones de evidencia para asegurar consistencia con el sistema ATC.


In [ ]:
# Limpieza de espacios y estandarización de strings
for c in df_canmat.columns:
    df_canmat[c] = df_canmat[c].astype(str).str.strip()

# Resumen de líneas de tratamiento
print("Distribución por línea de recomendación:")
print(df_canmat["linea"].value_counts())

print("\nDistribución por patología:")
print(df_canmat["patologia"].value_counts())


## 3. Generación de Reglas Clínicas para el Motor de Cascadas y RAG

Construimos una tabla derivada con alertas clínicas directas que se utilizarán para validar combinaciones potencialmente peligrosas o inapropiadas.


In [ ]:
alertas_canmat = []

for _, row in df_canmat[df_canmat["linea"].str.contains("No Recomendado", case=False)].iterrows():
    alertas_canmat.append({
        "patologia": row["patologia"],
        "farmaco_involucrado": row["farmaco"],
        "tipo_alerta": "Contraindicación / Riesgo de Viraje",
        "descripcion": row["comentarios"],
        "nivel_evidencia": row["nivel_evidencia"],
        "fuente": "Guías CANMAT / ISBD"
    })

df_alertas_canmat = pd.DataFrame(alertas_canmat)
df_alertas_canmat


## 4. Exportación de Resultados

Guardamos las recomendaciones estructuradas de CANMAT en la carpeta de datos procesados.


In [ ]:
out_canmat = DATA_PROCESSED / "canmat_recomendaciones_clean.csv"
out_alertas = DATA_PROCESSED / "canmat_alertas_clinicas_clean.csv"

df_canmat.to_csv(out_canmat, index=False, encoding="utf-8")
df_alertas_canmat.to_csv(out_alertas, index=False, encoding="utf-8")

print(f"Recomendaciones CANMAT guardadas en: {out_canmat}")
print(f"Alertas clínicas CANMAT guardadas en: {out_alertas}")
